!pip install torch tqdm scikit-learn matplotlib seaborn

In [1]:
import os
import numpy as np
import pandas as pd
#import rdata
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, roc_curve, f1_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import random
import seaborn as sns

In [2]:
# تنظیمات 
# ---------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", DEVICE)

Device: cpu


## فراخوانی داده و پیش پردازش

### فراخوانی داده

In [3]:
train_df = pd.read_csv('faulty_train.csv')
test_df = pd.read_csv('faulty_test.csv')

print("Faulty Train shape:", train_df.shape, "Faulty Test shape:", test_df.shape)
train_df.head()

Faulty Train shape: (150000, 55) Faulty Test shape: (242000, 55)


,faultNumber,simulationRun,sample,xmeas_1,xmeas_2,xmeas_3,xmeas_4,xmeas_5,xmeas_6,xmeas_7,...,xmv_2,xmv_3,xmv_4,xmv_5,xmv_6,xmv_7,xmv_8,xmv_9,xmv_10,xmv_11
0,7.0,42.0,107,0.44858,3582.6,4528.7,8.4448,26.758,41.772,2688.8,...,53.927,44.342,70.255,17.382,37.966,39.341,48.905,18.674,39.227,18.200
1,0.0,161.0,314,0.25968,3677.0,4534.7,9.3555,27.060,42.261,2693.9,...,53.680,25.747,60.924,21.498,39.539,29.402,43.028,50.196,41.186,20.000
2,0.0,105.0,87,0.27649,3707.7,4517.1,9.3591,27.109,42.101,2704.8,...,53.639,27.071,59.545,22.408,40.398,36.082,47.346,51.257,40.807,15.974
3,2.0,168.0,420,0.29588,3718.0,4656.3,9.5363,26.813,42.713,2691.9,...,55.380,29.328,62.577,20.712,81.510,42.843,46.549,24.862,41.627,16.144
4,9.0,93.0,233,0.25172,3719.8,4540.7,9.3783,27.083,42.526,2699.5,...,54.016,24.547,61.975,22.145,42.388,37.646,48.620,46.087,40.592,18.300


In [4]:
train_df['faultNumber'].unique()

array([ 7.,  0.,  2.,  9., 14., 20., 18.,  8., 12.,  3.,  5.,  4., 19.,
       13.,  6.,  1., 10., 15., 16., 17., 11.])

#### مجموعه دیتاست به دو بخش آموزش و ارزیابی از قبل تقسیم شده است. در هر بخش تعداد 20 نوع خرابی و 55 سنسور آورده شده است 

### پیش پردازش داده ها و ایجاد برچسب

In [5]:
def prepare_df(df):
    """تابع آماده سازی دیتافریم و استخراج ویژگی ها"""
    df = df.copy()
    cols = df.columns.tolist()
    
    # پیدا کردن ستون های مربوطه
    if 'faultNumber' in cols:
        fn_col = 'faultNumber'
    else:
        fn_col = cols[0]
    if 'simulationRun' in cols:
        sim_col = 'simulationRun'
    else:
        sim_col = cols[1]
    if 'sample' in cols:
        sample_col = 'sample'
    else:
        sample_col = cols[2]
    
    # ویژگی ها (ستون های 4 تا 55)
    feat_cols = cols[3:55] if len(cols) >= 55 else cols[3:]
    
    # تغییر نام ستون ها
    df = df.rename(columns={fn_col: 'faultNumber', sim_col: 'simulationRun', sample_col: 'sample'})
    
    # ایجاد ستون برچسب و اعمال شرط مقدار 0 برابر با نرمال و بیشتر از 0 غیرنرمال
    df['label'] = (df['faultNumber'] > 0).astype(int)

    return df, feat_cols

train_df, feat_cols = prepare_df(train_df)
test_df, _ = prepare_df(test_df)
print("تعداد ویژگی های مورد استفاده:", len(feat_cols))

# بررسی توزیع برچسب ها
print("\nتوزیع برچسب ها در داده های آموزش:")
print(train_df['label'].value_counts())
print("\nتوزیع برچسب ها در داده های ارزیابی:")
print(test_df['label'].value_counts())

# بررسی توزیع انواع خرابی
print("\nتوزیع نوع خرابی در داده های آموزش:")
print(train_df['faultNumber'].value_counts().sort_index())
print("\nتوزیع نوع خرابی در داده های ارزیابی:")
print(test_df['faultNumber'].value_counts().sort_index())

# استانداردسازی ویژگی ها
scaler = StandardScaler()
all_features = feat_cols
scaler.fit(pd.concat([train_df[all_features], test_df[all_features]], axis=0).values)

def df_to_xy(df):
    """تبدیل دیتافریم به آرایه های X و y"""
    X = scaler.transform(df[all_features].values)
    y = df['label'].values
    return X, y

X_train_full, y_train_full = df_to_xy(train_df)
X_test_full, y_test_full = df_to_xy(test_df)

# فقط نمونه های نرمال برای آموزش مدل های بدون ناظر
X_train_normal = X_train_full[y_train_full == 0]
print("نمونه های نرمال برای آموزش:", X_train_normal.shape)


تعداد ویژگی های مورد استفاده: 52

توزیع برچسب ها در داده های آموزش:
1    100016
0     49984
Name: label, dtype: int64

توزیع برچسب ها در داده های ارزیابی:
1    191912
0     50088
Name: label, dtype: int64

توزیع نوع خرابی در داده های آموزش:
0.0     49984
1.0      4955
2.0      4965
3.0      5033
4.0      4927
5.0      5070
6.0      4917
7.0      5108
8.0      4982
9.0      5031
10.0     5037
11.0     4931
12.0     5073
13.0     4995
14.0     4883
15.0     5000
16.0     5005
17.0     5096
18.0     4978
19.0     5024
20.0     5006
Name: faultNumber, dtype: int64

توزیع نوع خرابی در داده های ارزیابی:
0.0     50088
1.0      9718
2.0      9515
3.0      9534
4.0      9703
5.0      9598
6.0      9657
7.0      9656
8.0      9700
9.0      9420
10.0     9528
11.0     9735
12.0     9597
13.0     9524
14.0     9504
15.0     9542
16.0     9616
17.0     9686
18.0     9528
19.0     9670
20.0     9481
Name: faultNumber, dtype: int64
نمونه های نرمال برای آموزش: (49984, 52)


#### دیتاست اکنون به دو حالت نرمال و غیرنرمال تبدیل شده است. نوع خرابی لحاظ نشده است

## آموزش مدل ها
### مدل جنگل تصادفی به عنوان مدل نظارتی و مدل های Autoencoder و LSTM-AE به عنوان مدل های بدون ناظر آموزش داده می شوند

### مدل جنگل تصادفی

In [6]:
print("\nTraining Random Forest...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)
rf_model.fit(X_train_full, y_train_full)

# پیش بینی روی داده تست
rf_probs = rf_model.predict_proba(X_test_full)[:, 1]
rf_preds = rf_model.predict(X_test_full)

# محاسبه معیارهای ارزیابی
rf_auc = roc_auc_score(y_test_full, rf_probs)
rf_f1 = f1_score(y_test_full, rf_preds)

print(f"Random Forest AUC: {rf_auc:.4f}")
print(f"Random Forest F1 Score: {rf_f1:.4f}")


Training Random Forest...
Random Forest AUC: 0.8546
Random Forest F1 Score: 0.8052


### مدل Autoencoder

#### این مدل مبتنی بر شبکه عصبی است و برای آموزش آن از pytorch استفاده شده است.
#### از نمونه های نرمال نیز استفاده می شود زیرا مدل فقط رفتار نرمال را یاد بگیرد

In [ ]:
# PyTorch Dataset helpers
class TabularDataset(Dataset):
        # تبدیل numpy به PyTorch Dataset
        # برای استفاده در DataLoader 
    def __init__(self, X):
        self.X = torch.tensor(X, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx]

#  Autoencoder (FC) - PyTorch تعریف پارامترهای
class Autoencoder(nn.Module):
    def __init__(self, n_features, latent_dim=16):
        super().__init__()
        # تعریف نوع شبکه عصبی، تعداد نورون های ورودی و میانی، توابع فعال ساز و تعداد لایه ها
        self.encoder = nn.Sequential(
            nn.Linear(n_features, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, n_features)
        )
        # نحوه پیشروی انکدینگ و سپس دیکودینگ
    def forward(self, x):
        z = self.encoder(x)
        xhat = self.decoder(z)
        return xhat

def train_autoencoder(X_train_normal, X_val, n_features, epochs=80, batch_size=256, lr=1e-3):
    model = Autoencoder(n_features).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr) # استفاده از الگوریتم بهینه سازی Adam
    ds = TabularDataset(X_train_normal)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True)
    losses = []
    for e in range(epochs):
        model.train()
        epoch_loss = 0.0
        for xb in loader:
            xb = xb.to(DEVICE)
            xhat = model(xb)
            loss = nn.MSELoss()(xhat, xb)
            opt.zero_grad()
            loss.backward()
            opt.step()
            epoch_loss += loss.item() * xb.size(0)
        epoch_loss /= len(ds)
        losses.append(epoch_loss)
        if (e+1)%10==0 or e==0:
            print(f"AE Epoch {e+1}/{epochs} loss: {epoch_loss:.6f}")
    # محاسبه reconstruction error روی X_val
    model.eval()
    with torch.no_grad():
        Xv = torch.tensor(X_val, dtype=torch.float32).to(DEVICE)
        Xv_hat = model(Xv).cpu().numpy()
        recon_err = np.mean((Xv_hat - X_val)**2, axis=1)
    return model, losses, recon_err

print("\nآموزش Autoencoder روی نمونه های نرمال...")
ae_model, ae_losses, ae_test_recon_err = train_autoencoder(X_train_normal, X_test_full, n_features=X_train_full.shape[1], epochs=100)

# محاسبه AUC برای Autoencoder
ae_auc = roc_auc_score(y_test_full, ae_test_recon_err)
print(f"Autoencoder AUC: {ae_auc:.4f}")


آموزش Autoencoder روی نمونه های نرمال...
AE Epoch 1/100 loss: 0.130349
AE Epoch 10/100 loss: 0.023844
AE Epoch 20/100 loss: 0.023667
AE Epoch 30/100 loss: 0.023575
AE Epoch 40/100 loss: 0.023545
AE Epoch 50/100 loss: 0.023548
AE Epoch 60/100 loss: 0.023502


### LSTM Autoencoder مدل

####  از آنجایی که در دیتاست ستون های اجرای شبیه سازی و نمونه ها نیز وجود دارد، لذا خاصیت سری زمانی را دارا است. از اینرو از این مدل برای لحاظ نمودن سری زمانی استفاده می شود. 

### آماده سازی داده برای مدل LSTM Autoencoder

In [ ]:
def build_sequences_from_df(df, feat_cols, window=20, step=1):
    """تابع جهت تبدیل داده های جدولی به توالی زمانی برای LSTM"""
    seqs = []
    labels = []
    # گروه بندی بر اساس شماره شبیه سازی simulationRun
    for run, g in df.groupby('simulationRun'):
        g = g.sort_values('sample')  # مرتب سازی داده براساس زمان (ستون شماره نمونه)
        X = scaler.transform(g[feat_cols].values)
        y = g['label'].values
        #  ساخت پنجره زمانی برای مدل به طوری که اگر بیش از نیمی از مقادیر پنجره غیرنرمال باشد، مقدار آن squence را یک قرار می دهد
        for start in range(0, max(1, X.shape[0]-window+1), step):
            seq = X[start:start+window]
            lbl = int(y[start:start+window].mean() > 0.5)   #فقط وقتی کل پنجره fault=1 هست، برچسب 1می شود
            if seq.shape[0] == window:
                seqs.append(seq)
                labels.append(lbl)
    return np.array(seqs), np.array(labels)

print("\nساخت ترتیب برای LSTM...")
SEQ_LEN = 20  # اندازه پنجره پیشنهادی
X_seq_train, y_seq_train = build_sequences_from_df(train_df, feat_cols, window=SEQ_LEN)
X_seq_test, y_seq_test = build_sequences_from_df(test_df, feat_cols, window=SEQ_LEN)
print("ابعاد ترتیب آموزش:", X_seq_train.shape, "ابعاد ترتیب ارزیابی:", X_seq_test.shape)

print("تعداد ترتیب نرمال:", (y_seq_train == 0).sum())
print("تعداد ترتیب غیرنرمال:", (y_seq_train == 1).sum())

### تعریف پارامترهای مدل و آموزش

In [ ]:
class SeqDataset(Dataset): 
    '''تبدیل numpy به PyTorch Dataset
                برای استفاده در DataLoader
                                       '''
    def __init__(self, X):
        self.X = torch.tensor(X, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx]

class LSTMAE(nn.Module):
    def __init__(self, n_features, hidden_dim=32, latent_dim=16, n_layers=1):
        '''Sequence
               ↓
            Encoder LSTM
               ↓
            Latent vector
               ↓
            Decoder LSTM
               ↓
            Reconstructed Sequence'''
        super().__init__()
        # تعریف پارامترهای شبکه عصبی LSTMAutoEncoder
        self.encoder_lstm = nn.LSTM(n_features, hidden_dim, n_layers, batch_first=True)
        self.fc_enc = nn.Linear(hidden_dim, latent_dim)
        self.fc_dec = nn.Linear(latent_dim, hidden_dim)
        self.decoder_lstm = nn.LSTM(hidden_dim, hidden_dim, n_layers, batch_first=True)
        self.output_layer = nn.Linear(hidden_dim, n_features)
    def forward(self, x):
        enc_out, (h_n, c_n) = self.encoder_lstm(x)
        last = enc_out[:, -1, :]
        z = self.fc_enc(last)
        dec_in = self.fc_dec(z).unsqueeze(1).repeat(1, x.size(1), 1)
        dec_out, _ = self.decoder_lstm(dec_in)
        xhat = self.output_layer(dec_out)
        return xhat

def train_lstm_ae(X_train_seq, y_train_seq, X_val_seq, n_features, hidden_dim=32, latent_dim=16, n_layers=1, epochs=30, batch_size=64, lr=1e-3, patience=5):
    # فقط از نمونه های نرمال برای آموزش استفاده می شود
    X_train_normal = X_train_seq[y_train_seq == 0]
    
    model = LSTMAE(n_features, hidden_dim, latent_dim, n_layers).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    
    # استفاده از زیرمجموعه برای کاهش مصرف حافظه
    sample_indices = np.random.choice(len(X_train_normal), size=min(5000, len(X_train_normal)), replace=False)
    X_train_sampled = X_train_normal[sample_indices]
    
    ds = SeqDataset(X_train_sampled)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True)
    
    losses = []
    best_loss = float('inf')
    patience_counter = 0
    
    for e in range(epochs):
        model.train()
        epoch_loss = 0
        for xb in loader:
            xb = xb.to(DEVICE)
            xhat = model(xb)
            loss = nn.MSELoss()(xhat, xb)
            opt.zero_grad()
            loss.backward()
            opt.step()
            epoch_loss += loss.item() * xb.size(0)
        epoch_loss /= len(ds)
        losses.append(epoch_loss)
        
        # Early stopping
        if epoch_loss < best_loss:
            best_loss = epoch_loss
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {e+1}")
                break
        
        if (e+1)%5==0 or e==0:
            print(f"LSTM-AE Epoch {e+1}/{epochs} loss: {epoch_loss:.6f}")
    
    # محاسبه reconstruction error
    model.eval()
    with torch.no_grad():
        # استفاده از زیرمجموعه برای تست به دلیل محدودیت حافظه
        sample_indices_test = np.random.choice(len(X_val_seq), size=min(2000, len(X_val_seq)), replace=False)
        X_val_sampled = X_val_seq[sample_indices_test]
        y_val_sampled = y_seq_test[sample_indices_test]
        
        Xv = torch.tensor(X_val_sampled, dtype=torch.float32).to(DEVICE)
        Xv_hat = model(Xv).cpu().numpy()
        recon_err = np.mean((Xv_hat - X_val_sampled)**2, axis=(1,2)) #خطای بازسازی
    
    return model, losses, recon_err, y_val_sampled

print("\nآموزش LSTM Autoencoder روی توالی داده های آموزشی...")
lstm_model, lstm_losses, lstm_test_recon_err, y_val_sampled = train_lstm_ae(
    X_seq_train, y_seq_train, X_seq_test, 
    n_features=X_train_full.shape[1],
    hidden_dim=32,
    latent_dim=16,
    epochs=80,
    batch_size=64
)

# محاسبه AUC برای LSTM-AE
lstm_auc = roc_auc_score(y_val_sampled, lstm_test_recon_err)
print(f"LSTM-AE AUC: {lstm_auc:.4f}")

## تحلیل نتایج و مقایسه مدل ها

In [ ]:
def evaluate_model(y_true, y_pred, y_probs=None, model_name=""):
    """تابع جهت ارزیابی جامع مدل و نمایش نتایج"""
    print(f"\n=== نتایج ارزیابی برای {model_name} ===")
    
    # محاسبه معیارهای مختلف
    f1 = f1_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)
    
    print(f"F1 Score: {f1:.4f}")
    print("ماتریس درهم ریختگی:")
    print(cm)
    
    # نمایش ماتریس درهم ریختگی
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix - {model_name}')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()
    
    # رسم منحنی ROC اگر احتمالات موجود باشد
    if y_probs is not None:
        fpr, tpr, _ = roc_curve(y_true, y_probs)
        roc_auc = roc_auc_score(y_true, y_probs)
        
        plt.figure(figsize=(8, 6))
        plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
        plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.01])
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title(f'ROC Curve - {model_name}')
        plt.legend(loc="lower right")
        plt.show()
        
        return roc_auc, f1
    return None, f1

# ارزیابی Random Forest
rf_auc, rf_f1 = evaluate_model(y_test_full, rf_preds, rf_probs, "Random Forest")

# ارزیابی Autoencoder (تبدیل خطای بازسازی به پیش بینی دودویی)
ae_threshold = np.percentile(ae_test_recon_err, 95)  # استفاده از چندک 95ام به عنوان آستانه
ae_preds = (ae_test_recon_err > ae_threshold).astype(int)
ae_auc, ae_f1 = evaluate_model(y_test_full, ae_preds, ae_test_recon_err, "Autoencoder")

# ارزیابی LSTM-AE (تبدیل خطای بازسازی به پیش بینی دودویی)
lstm_threshold = np.percentile(lstm_test_recon_err, 95)  # استفاده از چندک 95ام به عنوان آستانه
lstm_preds = (lstm_test_recon_err > lstm_threshold).astype(int)
lstm_auc, lstm_f1 = evaluate_model(y_seq_test[:len(lstm_test_recon_err)], lstm_preds, lstm_test_recon_err, "LSTM-AE")

# مقایسه کلی مدل‌ها
print("\n=== مقایسه کلی مدل ها ===")
print("Model\t\tAUC\t\tF1 Score")
print("----------------------------------------")
print(f"Random Forest\t{rf_auc:.4f}\t\t{rf_f1:.4f}")
print(f"Autoencoder\t{ae_auc:.4f}\t\t{ae_f1:.4f}")
print(f"LSTM-AE\t\t{lstm_auc:.4f}\t\t{lstm_f1:.4f}")

### نمایش توزیع خطای بازسازی برای مدل های بدون ناظر Autoencoder

In [ ]:
# این نمودار توزیع تفاوت بین داده های ورودی و داده های بازسازی شده توسط مدل را نشان می دهد.
# مقدار آستانه نیز 95 درصد خطای بازسازی می باشد
def plot_error_distribution(errors, model_name, threshold):
    plt.figure(figsize=(10, 6))
    plt.hist(errors, bins=50, alpha=0.7, color='blue', edgecolor='black')
    plt.axvline(x=threshold, color='red', linestyle='--', label=f'Threshold: {threshold:.4f}')
    plt.xlim(0,)
    plt.xlabel('Reconstruction Error')
    plt.ylabel('Count')
    plt.title(f'Reconstruction Error Distribution - {model_name}')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

plot_error_distribution(ae_test_recon_err, "Autoencoder", ae_threshold)
plot_error_distribution(lstm_test_recon_err, "LSTM-AE", lstm_threshold)

#### برای مدل های بدون ناظر، از خطای بازسازی برای تشخیص ناهنجاری استفاده شده و آستانه بر اساس چندک 95ام تعیین شده است

In [ ]:
# تحلیل انواع خرابی ها
# ---------------------------
def analyze_fault_types(y_true, y_pred, fault_numbers, model_name):
    """تحلیل عملکرد مدل بر اساس انواع مختلف خرابی"""
    print(f"\n=== تحلیل عملکرد {model_name} بر اساس نوع خرابی ===")
    
    # ایجاد DataFrame برای تحلیل
    results_df = pd.DataFrame({
        'fault_number': fault_numbers,
        'true_label': y_true,
        'pred_label': y_pred
    })
    
    # گروه بندی بر اساس نوع خرابی
    fault_performance = results_df.groupby('fault_number').apply(
        lambda x: pd.Series({
            'samples': len(x),
            'true_anomalies': x['true_label'].sum(),
            'detected_anomalies': x['pred_label'].sum(),
            'detection_rate': x['pred_label'].sum() / max(1, x['true_label'].sum())
        })
    ).reset_index()
    
    print(fault_performance)
    
    # نمایش نموداری
    plt.figure(figsize=(12, 6))
    plt.bar(fault_performance['fault_number'].astype(str), fault_performance['detection_rate'])
    plt.xlabel('Fault Number')
    plt.ylabel('Detection Rate')
    plt.title(f'Detection Rate by Fault Type - {model_name}')
    plt.xticks(rotation=45)
    plt.grid(True, alpha=0.3)
    plt.show()
    
    return fault_performance

# تحلیل برای Random Forest
rf_fault_analysis = analyze_fault_types(y_test_full, rf_preds, test_df['faultNumber'].values, "Random Forest")

# تحلیل برای Autoencoder
ae_fault_analysis = analyze_fault_types(y_test_full, ae_preds, test_df['faultNumber'].values, "Autoencoder")

### ذخیره مدل ها و نتایج

In [ ]:
#  ذخیره مدل ها
torch.save(ae_model.state_dict(), "ae_model.pth")
torch.save(lstm_model.state_dict(), "lstm_ae_model.pth")

import joblib
joblib.dump(rf_model, "rf_model.joblib")
joblib.dump(scaler, "scaler.joblib")

# ذخیره نتایج
results_df = pd.DataFrame({
    'Model': ['Random Forest', 'Autoencoder', 'LSTM-AE'],
    'AUC': [rf_auc, ae_auc, lstm_auc],
    'F1_Score': [rf_f1, ae_f1, lstm_f1]
})
results_df.to_csv('model_comparison_results.csv', index=False)
print("مدل ها و نتایج بر روی دیسک ذخیره شد")